In [1]:
from transformers import DebertaV2ForTokenClassification, AutoTokenizer,  AdamW
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

class DebertaNER(nn.Module):
    def __init__(self, tokens_dim):
        super(DebertaNER, self).__init__()
        
        if type(tokens_dim) != int:
            raise TypeError('tokens_dim should be an integer')
        if tokens_dim <= 0:
            raise ValueError('Classification layer dimension should be at least 1')

        self.pretrained = DebertaV2ForTokenClassification.from_pretrained("microsoft/deberta-v3-large", num_labels=tokens_dim)

    def forward(self, input_ids, attention_mask, labels=None):
        """
        Forward pass of the network.
        """
        if labels is None:
            return self.pretrained(input_ids=input_ids, attention_mask=attention_mask)
        return self.pretrained(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

    def save_pretrained(self, save_directory):
        """
        Save the pretrained model to the specified directory.
        """
        self.pretrained.save_pretrained(save_directory)

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large", use_fast=True)

print("Tokenizer loaded successfully!")

/home/ritisha21089/.local/lib/python3.8/site-packages/transformers/convert_slow_tokenizer.py:558: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Tokenizer loaded successfully!


In [2]:
import torch
from torch.utils.data import Dataset
import ast

class NerDataset(Dataset):
    """
    Custom dataset class for NER tasks.
    """

    def __init__(self, dataframe, tokenizer, label_map, max_len=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.label_map = label_map
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]

        # Ensure the 'Ingredient_Phrases' column is a string
        text = str(row['Ingredient_Phrases'])

        # Convert 'Labels' from string representation to a list
        labels_list = ast.literal_eval(row['Labels'])  

        encoding = self.tokenizer(
            text.split(),  # Tokenize word by word
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
            is_split_into_words=True  # Ensure token alignment with words
        )

        word_ids = encoding.word_ids(batch_index=0)  # Get token-word alignment
        label_ids = [-100] * self.max_len  # Default to ignored token

        # Assigning NER labels for each word token
        previous_word_id = None
        for i, word_id in enumerate(word_ids):
            if word_id is None:
                label_ids[i] = -100  # Padding or special tokens
            elif word_id != previous_word_id:
                label_ids[i] = self.label_map.get(labels_list[word_id], -100)  # Assign the correct label
            else:
                label_ids[i] = label_ids[i-1]  # For subwords, keep same label
            previous_word_id = word_id

        input_ids = encoding['input_ids'].squeeze(0)  # Remove batch dimension
        attention_mask = encoding['attention_mask'].squeeze(0)  # Remove batch dimension
        label_ids = torch.tensor(label_ids, dtype=torch.long)  # Convert to tensor

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': label_ids
        }

# Label mapping 
label_map = {
    'QUANTITY': 0,
    'UNIT': 1,
    'NAME': 2,
    'FORM': 3,
    'STATE': 4,
    'DF': 5,
    'SIZE': 6,
    'O': -100  # Outside token or padding
}

from sklearn.model_selection import train_test_split
df = pd.read_csv('../../data/processed/10k_data_modified.csv')

- train + test split 

In [8]:
#ner_dataset = NerDataset(df, tokenizer, label_map)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = NerDataset(dataframe=train_df, tokenizer=tokenizer, label_map=label_map, max_len=128)
test_dataset = NerDataset(dataframe=test_df, tokenizer=tokenizer, label_map=label_map, max_len=128)

batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

- train test val split

In [3]:
#ner_dataset = NerDataset(df, tokenizer, label_map)
#ner_dataset = NerDataset(df, tokenizer, label_map)
train_df_, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_df_, test_size=0.2, random_state=42)

train_dataset = NerDataset(dataframe=train_df, tokenizer=tokenizer, label_map=label_map, max_len=128)
test_dataset = NerDataset(dataframe=test_df, tokenizer=tokenizer, label_map=label_map, max_len=128)
val_dataset = NerDataset(dataframe=val_df, tokenizer=tokenizer, label_map=label_map, max_len=128)

batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

In [6]:
import torch
torch.cuda.set_device(1)  # Set GPU 1 as the active device
print("Now using GPU:", torch.cuda.current_device())

Now using GPU: 1


In [9]:
model = DebertaNER(tokens_dim=len(label_map))
optimizer = AdamW(model.parameters(), lr=2e-5)

# Training settings
epochs = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.set_device(1) 
model.to(device)
model.train()

Some weights of DebertaV2ForTokenClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DebertaNER(
  (pretrained): DebertaV2ForTokenClassification(
    (deberta): DebertaV2Model(
      (embeddings): DebertaV2Embeddings(
        (word_embeddings): Embedding(128100, 1024, padding_idx=0)
        (LayerNorm): LayerNorm((1024,), eps=1e-07, elementwise_affine=True)
        (dropout): StableDropout()
      )
      (encoder): DebertaV2Encoder(
        (layer): ModuleList(
          (0-23): 24 x DebertaV2Layer(
            (attention): DebertaV2Attention(
              (self): DisentangledSelfAttention(
                (query_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (key_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (value_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (pos_dropout): StableDropout()
                (dropout): StableDropout()
              )
              (output): DebertaV2SelfOutput(
                (dense): Linear(in_features=1024, out_features=1024, bias=True)
 

# MODEL TRAINING

- validation + training 

In [ ]:
import matplotlib.pyplot as plt
from tqdm import tqdm  # Import tqdm for progress bars

# Lists to store losses
train_losses = []
val_losses = []

for epoch in range(epochs):
    total_train_loss = 0
    total_val_loss = 0

    # Training phase
    model.train()
    train_iterator = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Training]")  

    for batch in train_iterator:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        train_iterator.set_postfix({"Train Loss": f"{loss.item():.4f}"})  # Update tqdm bar

    avg_train_loss = total_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # Validation phase (no gradient updates)
    model.eval()
    val_iterator = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Validation]", leave=False)

    with torch.no_grad():
        for batch in val_iterator:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            total_val_loss += loss.item()
            val_iterator.set_postfix({"Val Loss": f"{loss.item():.4f}"})  # Update tqdm bar

    avg_val_loss = total_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

# Plot Training vs Validation Loss Curve
plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), train_losses, label="Train Loss", marker='o', linestyle='-')
plt.plot(range(1, epochs + 1), val_losses, label="Validation Loss", marker='o', linestyle='-')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss Curve")
plt.legend()
plt.grid()
plt.show()

- early stopping + training

In [10]:
import torch
import numpy as np
from tqdm import tqdm

# Train
train_losses = []
val_losses = []
patience = 3  # Stop training if validation loss does not improve for 'patience' epochs
best_val_loss = np.inf  # Initialize best validation loss as infinity
early_stop_counter = 0  # Tracks epochs without improvement
epochs = 8

for epoch in range(epochs):
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)

    # Training Loop
    model.train()
    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Update tqdm description with latest loss
        progress_bar.set_postfix(loss=loss.item())

    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)  # Store training loss

    # Validation Step
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:  # Assuming you have a validation DataLoader
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)  # Store validation loss

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Validation Loss: {avg_val_loss:.4f}")

    # Early Stopping Check
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss  # Update best validation loss
        early_stop_counter = 0  # Reset counter
        torch.save(model.state_dict(), "best_model.pth")  # Save best model
    else:
        early_stop_counter += 1
        print(f"Early stopping counter: {early_stop_counter}/{patience}")

    if early_stop_counter >= patience:
        print("Early stopping triggered! Stopping training.")
        break  # Stop training

Epoch 1/8:   0%|          | 0/432 [00:00<?, ?it/s]

Epoch 1/8, Train Loss: 0.1461, Validation Loss: 0.0813


Epoch 2/8, Train Loss: 0.0564, Validation Loss: 0.0716


Epoch 3/8, Train Loss: 0.0476, Validation Loss: 0.0779
Early stopping counter: 1/3


Epoch 4/8, Train Loss: 0.0308, Validation Loss: 0.0828
Early stopping counter: 2/3


Epoch 5/8, Train Loss: 0.0221, Validation Loss: 0.0948
Early stopping counter: 3/3
Early stopping triggered! Stopping training.


- cross fold validation

In [ ]:
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold  # To check for imbalance
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm  # Import tqdm for progress bars

k_folds = 5  # Number of folds
batch_size = 16
epochs = 5
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

# Convert DataFrame to NumPy for indexing
df_np = df.to_numpy()
fold_results = []

# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# for fold, (train_idx, val_idx) in enumerate(skf.split(df_np, df["Labels"])):  # Use your label column

for fold, (train_idx, val_idx) in enumerate(kf.split(df_np)):
    print(f"\n{'-'*30}\nFold {fold+1}/{k_folds}\n{'-'*30}")

    train_df = pd.DataFrame(df_np[train_idx], columns=df.columns)
    val_df = pd.DataFrame(df_np[val_idx], columns=df.columns)

    # Create datasets
    train_dataset = NerDataset(dataframe=train_df, tokenizer=tokenizer, label_map=label_map, max_len=128)
    val_dataset = NerDataset(dataframe=val_df, tokenizer=tokenizer, label_map=label_map, max_len=128)

    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Initialize model
    model = BertNER(tokens_dim=len(label_map)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

    # Training loop
    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        total_train_loss = 0
        total_val_loss = 0

        # Training phase with tqdm
        model.train()
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Training]", leave=False)

        for batch in train_bar:
            optimizer.zero_grad()

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            train_bar.set_postfix(loss=loss.item())

        avg_train_loss = total_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation phase with tqdm
        model.eval()
        all_preds = []
        all_labels = []

        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Validation]", leave=False)
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                logits = outputs.logits  # Ensure logits are present

                predictions = torch.argmax(logits, dim=-1)
                
                # Store predictions and labels
                all_preds.extend(predictions.cpu().numpy().flatten())  
                all_labels.extend(labels.cpu().numpy().flatten())  

                loss = outputs.loss
                total_val_loss += loss.item()
    
        avg_val_loss = total_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

    # Print classification report
    print("\nClassification Report for Fold:")
    print(classification_report(all_labels, all_preds, digits=4))

    fold_results.append({
        'train_losses': train_losses, 
        'val_losses': val_losses,
        'classification_report': classification_report(all_labels, all_preds, digits=4, output_dict=True)
    })

# Aggregate results from all folds
avg_train_loss_per_epoch = np.mean([fold['train_losses'] for fold in fold_results], axis=0)
avg_val_loss_per_epoch = np.mean([fold['val_losses'] for fold in fold_results], axis=0)

# Plotting the loss curve
plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), avg_train_loss_per_epoch, label="Avg Train Loss", marker='o', linestyle='-')
plt.plot(range(1, epochs + 1), avg_val_loss_per_epoch, label="Avg Validation Loss", marker='o', linestyle='-')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title(f"{k_folds}-Fold Cross-Validation Loss Curve")
plt.legend()
plt.grid()
plt.show()

- Training

In [10]:
from tqdm import tqdm

# Train
train_losses = []

for epoch in range(epochs):
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)

    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Update tqdm description with the latest loss
        progress_bar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)  # Store loss

    print(f"Epoch {epoch+1}/{epochs}, Average Loss: {avg_loss:.4f}")

Epoch 1/5, Average Loss: 0.1331


Epoch 2/5, Average Loss: 0.0565


Epoch 3/5, Average Loss: 0.0557


Epoch 4/5, Average Loss: 0.0301


Epoch 5/5, Average Loss: 0.0230


# EVALUATION

In [11]:
# Evaluation 
model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        logits = logits.detach().cpu().numpy()
        label_ids = labels.cpu().numpy()

        for i in range(len(label_ids)):
            pred_labels = []
            true_label = []
            for j in range(len(label_ids[i])):
                if label_ids[i][j] != -100:
                    true_label.append(label_ids[i][j])
                    pred_label = np.argmax(logits[i][j])
                    pred_labels.append(pred_label)
            predictions.append(pred_labels)
            true_labels.append(true_label)

In [7]:
from sklearn.metrics import classification_report

# Convert to flat lists for metrics calculation
flat_predictions = [item for sublist in predictions for item in sublist]
flat_true_labels = [item for sublist in true_labels for item in sublist]

precision = precision_score(flat_true_labels, flat_predictions, average='macro')
recall = recall_score(flat_true_labels, flat_predictions, average='macro')
f1 = f1_score(flat_true_labels, flat_predictions, average='macro')

print("DeBERTa-v3-large results: ")
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

report = classification_report(flat_true_labels, flat_predictions, digits=4)
print(report)

DeBERTa-v3-large results: 
Precision: 0.9769
Recall: 0.9796
F1-Score: 0.9782
              precision    recall  f1-score   support

           0     0.9994    0.9978    0.9986      3637
           1     0.9959    0.9964    0.9962      1690
           2     0.9837    0.9813    0.9825      3746
           3     0.9369    0.9209    0.9288       822
           4     0.9769    0.9836    0.9802      2189
           5     0.9813    0.9953    0.9882       211
           6     0.9645    0.9819    0.9731       553

    accuracy                         0.9847     12848
   macro avg     0.9769    0.9796    0.9782     12848
weighted avg     0.9847    0.9847    0.9847     12848



# MODEL SAVING 

In [ ]:
# import torch

# # Define checkpoint path
# checkpoint_path = "../../models/deberta_v3_large/deberta_v3_large_ner.pth"

# # Save the model and optimizer state
# torch.save({
#     'epoch': epochs,
#     'model_state_dict': model.state_dict(),
#     'optimizer_state_dict': optimizer.state_dict(),
# }, checkpoint_path)

# print(f"Checkpoint saved at {checkpoint_path}")

Checkpoint saved at deberta_v3_large_ner.pth


In [ ]:
# import pandas as pd

# df_results = pd.DataFrame({
#     "True_Label": flat_true_labels,
#     "Predicted_Label": flat_predictions
# })

# df_results.to_csv("../../results/deberta_v3_large_ner_predictions.csv", index=False)
# print("Predictions saved to deberta_v3_large_ner_predictions.csv!")

Predictions saved to deberta_v3_large_ner_predictions.csv!


In [ ]:
# import torch

# # Define test dataset save path
# test_dataset_path = "../../models/deberta_v3_large/test_loader_deberta_v3_large.pth"

# # Save the dataset (which is used in test_loader)
# torch.save(test_dataset, test_dataset_path)

# print(f"Test dataset saved at {test_dataset_path}")

Test dataset saved at test_loader_deberta_v3_large.pth


# Loading model

In [4]:
import torch
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# Define checkpoint path
checkpoint_path = "../../models/deberta_v3_large/deberta_v3_large_ner.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DebertaNER(tokens_dim=len(label_map))
optimizer = AdamW(model.parameters(), lr=2e-5)

# Load the checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)


# Load model and optimizer state
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# Ensure model is in evaluation mode
model.to(device)
model.eval()

print("Model loaded successfully and set to evaluation mode!")

Some weights of DebertaV2ForTokenClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/ritisha21089/miniconda3/lib/python3.9/site-packages/transformers/optimization.py:640: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
/tmp/ipykernel_2927919/4265313921.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models f

Model loaded successfully and set to evaluation mode!


In [5]:
predictions = []
true_labels = []

batch_size = 16
test_dataset_path = "../../models/deberta_v3_large/test_loader_deberta_v3_large.pth"
test_dataset = torch.load(test_dataset_path)
test_loader = DataLoader(test_dataset, batch_size=batch_size)
print("Test loader reloaded successfully!")

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        logits = logits.detach().cpu().numpy()
        label_ids = labels.cpu().numpy()

        for i in range(len(label_ids)):
            pred_labels = []
            true_label = []
            for j in range(len(label_ids[i])):
                if label_ids[i][j] != -100:  # Ignore padding labels
                    true_label.append(label_ids[i][j])
                    pred_label = np.argmax(logits[i][j])
                    pred_labels.append(pred_label)
            predictions.append(pred_labels)
            true_labels.append(true_label)

flat_predictions_new = [item for sublist in predictions for item in sublist]
flat_true_labels_new = [item for sublist in true_labels for item in sublist]

precision = precision_score(flat_true_labels_new, flat_predictions_new, average='macro')
recall = recall_score(flat_true_labels_new, flat_predictions_new, average='macro')
f1 = f1_score(flat_true_labels_new, flat_predictions_new, average='macro')

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

report = classification_report(flat_true_labels_new, flat_predictions_new, digits=4)
print(report)

/tmp/ipykernel_2927919/4281528962.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  test_dataset = torch.load(test_dataset_path)


Test loader reloaded successfully!
Precision: 0.9769
Recall: 0.9796
F1-Score: 0.9782
              precision    recall  f1-score   support

           0     0.9994    0.9978    0.9986      3637
           1     0.9959    0.9964    0.9962      1690
           2     0.9837    0.9813    0.9825      3746
           3     0.9369    0.9209    0.9288       822
           4     0.9769    0.9836    0.9802      2189
           5     0.9813    0.9953    0.9882       211
           6     0.9645    0.9819    0.9731       553

    accuracy                         0.9847     12848
   macro avg     0.9769    0.9796    0.9782     12848
weighted avg     0.9847    0.9847    0.9847     12848

